# Take-Home Case: The Mortgage Book of Aare-Säntis Regionalbank AG

**EAIF: AI for Finance. Team take-home between the classes on linear/logistic regression and on advanced supervised learning**

You are the newly formed data team of Aare-Säntis Regionalbank AG, a fictional regional bank in the Swiss Mittelland. The bank has 10,000 residential mortgages on its books, originated between 2019 and 2022, spread over seven cantons. Until now, the bank's credit decisions have rested on a vendor valuation model, a handful of ratio rules and the judgement of the branch advisers. This morning the Chief Risk Officer sent the team its first assignment.

> **Memo from the CRO**
>
> To the data team. Welcome aboard. Three things, in order of urgency.
>
> 1. Our collateral valuations come from a vendor model we cannot inspect. Build one we can.
> 2. About seven in a hundred of our recent mortgages ran into payment trouble within three years. Tell me which applications carry that risk, and where we should set the approval bar, in francs.
> 3. The risk committee has read that banks use "machine learning" for this. Show me whether the more flexible methods actually do better on our book, and whether we could defend deploying one.
>
> I do not need a slide deck. I need numbers I can trust and one paragraph per question that I can read out to the committee.
>
> Head of Risk

This notebook is your working file for the assignment. Part A answers the first two requests with the methods you already know, linear and logistic regression. Part B answers the third with the methods introduced in the next class. Every section ends with what you, the analyst, tell the CRO.

## The data

Two files are loaded from the course repository. `mortgages.csv` is the bank's book: 10,000 mortgages originated 2019 to 2022, each with its outcome after 36 months. `applications_2026.csv` holds this year's 500 applications. It has the same columns as the book except two: the outcome, which does not exist yet because the bank has not decided on them, and one further column that Part A2 comes to. The applications file has 22 columns where the book has 24. The full column dictionary is in [`data/mortgage2026/README.md`](https://github.com/umatter/EDFB/blob/main/data/mortgage2026/README.md); the groups of columns are these.

| Group | Columns |
|---|---|
| Property | `canton`, `property_type`, `living_area_m2`, `rooms`, `year_built`, `distance_center_km`, `energy_label`, `purchase_price` |
| Borrower | `household_income`, `age`, `employment`, `years_client` |
| Loan | `loan_amount`, `rate_type`, `fixed_years`, `interest_rate`, `amortisation`, `origination_year` |
| Derived ratios | `ltv`, `affordability`, `actual_burden` |
| Outcome | `trouble_36m` (1 if the mortgage was 90 days or more in arrears, or was restructured, within 36 months) |

Three ratios do most of the work in Swiss mortgage lending, and all three are in the file.

The loan-to-value ratio compares the loan with the price of the property: `ltv = loan_amount / purchase_price`. Swiss banks normally finance at most 80 % of the price, and the part of the loan above two-thirds of the price must be amortised within 15 years.

The affordability ratio is the Swiss lending rule for whether the household can carry the loan through a rise in interest rates. It does not use the interest rate actually agreed but an imputed rate of 5 %, adds 1 % of the purchase price per year for maintenance, adds the amortisation of the part above two-thirds LTV spread over 15 years, and divides the sum by gross household income:

`affordability = (0.05 × loan_amount + 0.01 × purchase_price + amortisation per year) / household_income`

The rule says the ratio must not exceed one third. A household with CHF 150,000 gross income and a CHF 800,000 loan on a CHF 1,000,000 property carries 40,000 of imputed interest, 10,000 of maintenance and about 8,900 of amortisation per year, which is 0.39 of its income, above the bar.

The actual burden is what the household pays in interest today, at the agreed rate: `actual_burden = interest_rate × loan_amount / household_income`. With rates mostly between 1 and 2 %, it is a fraction of the affordability ratio, which is exactly why the imputed rate exists.

> **Simulated data.** The two files were generated by the course for this case. There is no real bank, property or household behind any row. The rate of payment trouble in the book is several times higher than in a real Swiss mortgage book so that the models have enough cases to learn from; the ratios, prices and incomes are in a realistic range but were not taken from any real data source.

One more thing to know before you start: the book contains one column that the bank does *not* know at the moment it decides on an application. Part A2 deals with it.

## How to work through this notebook

Plan three hours for a team of three to four. Make a copy of the notebook in your Drive (File, Save a copy in Drive) and work in the copy. Run the given cells one after the other and read them, including the comments; they are the worked part of the case and they are where the methods are explained. The exercises are marked `### Exercise n` and each one is followed by a code cell that holds only a comment. Fill that cell and leave the given cells as they are, because later parts of the notebook use the objects they create.

Rotate who types for each part, so that nobody sits through the whole case as a spectator. In the next class, any member of the team can be asked to explain any cell, worked or exercise, so make sure everyone can. The notebook is discussed in that class; it is not graded.

Every choice of columns in this case follows one rule, which you will meet again in every part:

**At the moment the bank decides on an application, which columns are already known?**

Everything a model uses must pass that test. A model that is fed a column the bank only learns afterwards looks excellent in the notebook and is useless at the counter.

## Roadmap

| Part | Question | Method | Exercises |
|---|---|---|---|
| A1 | Is our collateral valued right? | Linear regression | 1, 2 |
| A2 | Which applications will run into payment trouble, and where is the approval bar in francs? | Logistic regression | 3, 4 |
| B | Do the flexible methods do better on our book? | LASSO, decision tree, random forest, gradient boosting and XGBoost, SVM, comparison | |
| C | Your turn on the new methods | The methods of Part B, and reflection questions for the class | 5 to 8 |

## Setup

In [ ]:
# Setup: Colab's preinstalled stack only, nothing to install
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance, DecisionBoundaryDisplay
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score, roc_auc_score,
                             roc_curve, confusion_matrix, precision_score, recall_score)
import xgboost as xgb

np.random.seed(0)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
sns.set_theme(style="whitegrid")

# The two error costs the CRO gave us, in CHF, used throughout the notebook
COST_FN = 60_000   # a loan that runs into trouble was approved: expected loss
COST_FP = 8_000    # a loan that would have been fine was rejected: lost margin

print("Setup complete. scikit-learn", __import__("sklearn").__version__, "| xgboost", xgb.__version__)

In [ ]:
# Load the two data files from the course repository
DATA = "https://raw.githubusercontent.com/umatter/EDFB/main/data/mortgage2026/"
book = pd.read_csv(DATA + "mortgages.csv")
apps = pd.read_csv(DATA + "applications_2026.csv")
print("book:", book.shape, "| applications:", apps.shape)
book.head()

In [ ]:
# Types, missing values, and the outcome's base rate
print(book.dtypes, "\n")
print("missing values:", int(book.isna().sum().sum()))
print(f"trouble rate in the book: {book.trouble_36m.mean():.3%}  ({book.trouble_36m.sum()} of {len(book)})")
book.describe().T

# Part A: Recap

## A1. Is our collateral valued right? Linear regression

The bank lends against the property. If the borrower stops paying, the bank sells the property and recovers what it can, so the question behind every mortgage is what the property is worth, as opposed to what the buyer paid for it. Today that question is answered by a vendor model that returns a number and no explanation. The CRO's first request is a valuation the bank can inspect.

The standard tool for this is a hedonic model: the price of a property is written as the sum of the prices of its characteristics. A linear regression is exactly such a model, and it is inspectable by construction. Its coefficients are a price per square metre, a premium or discount per canton, a discount per kilometre from the regional centre, a premium for a house over an apartment. A valuer can read those numbers, argue with them, and compare them with what the market pays.

We follow the steps of the first supervised-learning class.

1. Look at the data, in a plot, before fitting anything.
2. Pick the target Y (`purchase_price`) and the features X (the property columns).
3. Split the book into a training set and a test set.
4. Fit the model on the training set.
5. Test it on the held-out data, and put its RMSE next to the RMSE of a baseline that knows nothing.

The features are all property characteristics, which the bank knows when the application arrives, so the decision-time rule is satisfied.

In [ ]:
# Step 1: look at the relationship we want to model. Price against living area, one point per mortgage.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(book.living_area_m2, book.purchase_price / 1e6, s=6, alpha=0.3)
ax.set_xlabel("living area (m²)")
ax.set_ylabel("purchase price (CHF million)")
ax.set_title("The bank's book: purchase price against living area")
plt.show()

In [ ]:
# Step 2: the simplest model. One X, one slope: CHF per square metre, averaged over everything else.
uni = LinearRegression().fit(book[["living_area_m2"]], book.purchase_price)
print(f"price = {uni.intercept_:,.0f} + {uni.coef_[0]:,.0f} x living area")
print(f"R² on the full book: {uni.score(book[['living_area_m2']], book.purchase_price):.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(book.living_area_m2, book.purchase_price / 1e6, s=6, alpha=0.3)
grid = np.linspace(45, 320, 50).reshape(-1, 1)
ax.plot(grid, uni.predict(pd.DataFrame(grid, columns=["living_area_m2"])) / 1e6, color="C3", lw=2)
ax.set_xlabel("living area (m²)"); ax.set_ylabel("purchase price (CHF million)")
ax.set_title("Simple linear regression: one slope for all cantons")
plt.show()

The slope is the whole model. In this run it says that one additional square metre of living area adds CHF 7,589 to the price, averaged over every canton, every building age and every location in the book, and living area alone explains 48 % of the variation in prices (an R² of 0.484). The line runs through the middle of the cloud, but the cloud is wide: at 150 m² the book holds properties that sold for well under one million and others that sold for close to two.

One slope is not enough because a square metre does not cost the same everywhere. A square metre in the canton of Zurich and one in the canton of Solothurn are priced in different markets. A house built in 1960 and one built in 2018 differ in what a buyer pays, as do a flat next to the station and one twenty kilometres out. The simple model averages over all of that, and the spread around the line is the price of averaging.

The multivariate model puts these characteristics in as further columns of X. The numeric ones (`living_area_m2`, `year_built`, `distance_center_km`) enter as they are. The categorical ones (`canton`, `property_type`, `energy_label`) become 0/1 dummy columns, one per level, with one reference level dropped per column, exactly as in the logistic-regression class: the coefficient of each dummy is then the premium relative to the dropped level. `pd.get_dummies(..., drop_first=True)` drops the alphabetically first level, so the references are canton AG, property type `apartment` and energy label A.

In [ ]:
# Step 3: the multivariate model. Categorical columns become 0/1 dummies (one reference level dropped each).
price_features = ["canton", "property_type", "living_area_m2", "year_built", "distance_center_km", "energy_label"]
Xp = pd.get_dummies(book[price_features], drop_first=True).astype(float)
yp = book.purchase_price

Xp_train, Xp_test, yp_train, yp_test = train_test_split(Xp, yp, test_size=0.3, random_state=42, stratify=book.canton)
lin = LinearRegression().fit(Xp_train, yp_train)

coef = pd.Series(lin.coef_, index=Xp.columns).sort_values()
print("intercept:", f"{lin.intercept_:,.0f}")
coef.round(0).to_frame("CHF per unit")

Every row of the table is a price the bank can read. In this run, one square metre of living area adds CHF 6,705, now holding canton, building age, location and energy label fixed, which is about CHF 900 less than the slope of the simple model. The canton dummies are premiums relative to Aargau, the dropped reference: a property in the canton of Zurich sells for about CHF 403,000 more than the same property in Aargau, one in Solothurn for about CHF 187,000 less. Each kilometre further from the regional centre takes about CHF 17,900 off the price. A house sells for about CHF 96,000 more than an apartment with the same area, age, canton and location, and each energy label below A carries its own discount.

These numbers are the inspectable model the CRO asked for. A valuer who disagrees with the Zurich premium or the discount per kilometre can say so, in francs, and the bank can check the number against recent transactions.

In [ ]:
# Step 4: test the model on data it has not seen, next to the baseline "predict the training mean"
def rmse_of(y, pred):
    return float(np.sqrt(mean_squared_error(y, pred)))

pred_test = lin.predict(Xp_test)
baseline = np.full(len(yp_test), yp_train.mean())
print(f"RMSE  linear model : CHF {rmse_of(yp_test, pred_test):>10,.0f}   (R² {r2_score(yp_test, pred_test):.3f})")
print(f"RMSE  predict mean : CHF {rmse_of(yp_test, baseline):>10,.0f}   (R² {r2_score(yp_test, baseline):.3f})")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(yp_test / 1e6, pred_test / 1e6, s=6, alpha=0.3)
lim = [0, yp_test.max() / 1e6]
axes[0].plot(lim, lim, color="C3", lw=1)
axes[0].set_xlabel("actual price (CHF million)"); axes[0].set_ylabel("predicted price (CHF million)")
axes[0].set_title("Test set: predicted against actual")
axes[1].scatter(pred_test / 1e6, (yp_test - pred_test) / 1e6, s=6, alpha=0.3)
axes[1].axhline(0, color="C3", lw=1)
axes[1].set_xlabel("predicted price (CHF million)"); axes[1].set_ylabel("residual (CHF million)")
axes[1].set_title("Residuals fan out as the price grows")
plt.show()

The test set holds 3,000 mortgages the model has never seen. In this run the linear model's RMSE on the test set is CHF 154,786, against CHF 426,031 for the baseline that predicts the training mean for every property, and it explains 87 % (an R² of 0.868) of the variation in test prices where the baseline explains none. For a property that the model values at CHF 900,000, an error of one RMSE means that the price such a property fetches is plausibly anywhere between about CHF 745,000 and CHF 1,055,000, which is the uncertainty the bank has to keep in mind when it sets the loan-to-value ratio.

The right-hand plot shows something the RMSE hides. The residuals are not a band of constant width; they fan out. Cheap properties are missed by a small amount, expensive ones by a large amount, and the error grows in proportion to the price. That is the signature of a multiplicative relationship: a Zurich premium is more naturally a percentage of the price than a fixed sum in francs, and so is the discount for an old building. A model in levels cannot express that, which is why it is imprecise exactly where the bank's exposures are largest. Exercise 1 takes the hint.

What the analyst tells the CRO: a linear regression on six property characteristics values the collateral to within about CHF 155,000 on unseen properties, every coefficient is a price in francs that a valuer can inspect and challenge, and the model's main weakness, its imprecision for expensive properties, can be addressed by modelling the price in logarithms, which is the next step.

### Exercise 1: A model in logarithms

Fit the same multivariate model on `np.log(yp_train)` instead of `yp_train`. Predict on the test set, transform the predictions back with `np.exp`, and report the RMSE in CHF next to the RMSE of the model in levels. Then plot the residuals of the log model against its predictions as in the cell above.

*Deliverable:* the two RMSE values, and one sentence on which model the bank should use and why (look at the residual plot, not only at the RMSE).

In [ ]:
# Exercise 1: your code here
# lin_log = ...

### Exercise 2: Did the buyer overpay? A feature for Part B

Use your log model to predict a value for **every** property in the book (`Xp`, not only the test set) and compute `overpayment = purchase_price / predicted value`. A value above 1 means the buyer paid more than comparable properties cost. Store it as `book["overpayment"]`. Then compare the trouble rate of the mortgages in the top 10 % of `overpayment` with the rest.

*Deliverable:* the two trouble rates and one sentence on whether the valuation model tells the bank something about repayment risk. (Part B recomputes this column itself, so the notebook keeps running if you skip this.)

In [ ]:
# Exercise 2: your code here
# book["overpayment"] = ...

## A2. Which borrowers run into payment trouble? Logistic regression

The CRO's second request is different in kind from the first. The target is no longer a price but a yes or no: did the mortgage run into payment trouble within 36 months (`trouble_36m` equal to 1) or not. This is classification, and the tool from the second supervised-learning class is logistic regression. It does not predict the outcome itself; it predicts a probability of trouble for each application, and the bank turns that probability into a decision by choosing a threshold above which it declines. The threshold is where the CRO's phrase "in francs" enters, and it is the subject of the second half of this part.

We follow the sequence of that class.

1. Look at the base rate, and at what a model that always predicts the majority class would score.
2. Pick the features, applying the decision-time rule column by column.
3. Split the book into a training and a test set, stratified so that both hold the same share of troubled loans, and standardise the features on the training set only.
4. Fit the logistic regression and read its coefficients as odds ratios.
5. Turn the test-set probabilities into decisions at a threshold and read the confusion matrix.
6. Judge the ranking independently of any threshold with the ROC curve and its AUC, computed from the probabilities.
7. Put a price on the two kinds of error, in CHF, and ask where the threshold should sit.

The decision-time rule bites here for the first time. The column dictionary marks one column of the book as known only afterwards: `reminders_sent`, the number of payment reminders the bank sent during the 36 months. It is the column that the applications file does not have, and the cell below shows why it cannot be a predictor: it is recorded after the outcome it would predict, and it is to a large extent the same event seen from the bank's side. Three further columns are known at decision time but are left out for the reasons the comment gives. `purchase_price` and `loan_amount` already enter through the three ratios, and `origination_year` takes a value in the applications, 2026, that the book never contains, so a coefficient estimated on 2019 to 2022 could not be applied to it. A fourth, `overpayment`, exists only if your team completed Exercise 2; the cell leaves it out so that every team fits the same model here, and Part B brings it in.

In [ ]:
# Which columns does the bank know when it decides? Everything in the file except the outcome and one more.
# reminders_sent counts the payment reminders sent DURING the 36 months: it is a consequence of trouble,
# not a predictor. A model that uses it looks brilliant on the book and is useless for an application.
print(book.groupby("trouble_36m").reminders_sent.mean().rename("mean reminders_sent"))
print()
# purchase_price and loan_amount are known, but they enter through ltv / affordability / actual_burden;
# origination_year is known, but this year's applications come from 2026, a year the book never saw.
# overpayment, if Exercise 2 created it, is left for Part B, so that every team sees the same numbers here.
target = "trouble_36m"
not_features = ["id", "purchase_price", "loan_amount", "origination_year", "reminders_sent", "overpayment", target]
features = [c for c in book.columns if c not in not_features]
print("features used:", features)

In [ ]:
# Base rate and the majority-class baseline: a "model" that approves everyone is right 93% of the time
print(f"trouble rate: {book[target].mean():.3%}")
print(f"accuracy of 'nobody gets into trouble': {1 - book[target].mean():.3%}")

In [ ]:
# Split first (stratified on the outcome), then standardise on the training set only
X = pd.get_dummies(book[features], drop_first=True).astype(float)
y = book[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=X.columns, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)
print("train:", X_train.shape, "| test:", X_test.shape, "| columns:", list(X.columns))

In [ ]:
# Logistic regression. Coefficients are log-odds per one standard deviation of the feature (we standardised),
# and exp(coef) is the odds ratio: how much the odds of trouble multiply when the feature rises by one SD.
logit = LogisticRegression(max_iter=2000).fit(X_train_s, y_train)
p_logit = logit.predict_proba(X_test_s)[:, 1]

odds = pd.DataFrame({"coef (log-odds per SD)": logit.coef_[0], "odds ratio": np.exp(logit.coef_[0])},
                    index=X.columns).sort_values("coef (log-odds per SD)", key=abs, ascending=False)
odds.round(3)

Every row of the table is an odds ratio, and an odds ratio is a multiplier on the odds of trouble when the feature rises by one standard deviation, holding the other columns fixed. In this run the largest is `affordability`: one standard deviation more, which is about 0.13 more of gross income needed to carry the imputed cost, multiplies the odds of trouble by 2.7. The Swiss rule looks at the right ratio. Next come `household_income` with 1.7 and `rate_type_saron` with 1.7, then `ltv` with 1.6, where one standard deviation is about ten points of loan-to-value.

Two of these need care. The income row points the way a reader may not expect: more income, higher odds of trouble. It is read holding the three ratios fixed, and at the same affordability and loan-to-value more income means a larger loan, so the row says that larger exposures at the same ratios go wrong more often, not that the bank should prefer poorer households. A coefficient on a column that also sits in the denominator of three other features is the clearest warning in the table against reading any row on its own. The SARON row is a 0/1 column, so one standard deviation is not a switch from a fixed to a variable rate; that switch is about two standard deviations, and the odds of trouble for a SARON mortgage are about 2.9 times those of a comparable fixed-rate one.

An odds ratio below 1 is protective. The Zurich dummy has an odds ratio of 0.7 per standard deviation, and as with SARON the full switch from Aargau to Zurich is a larger effect in the same direction: a comparable property in Zurich carries clearly lower odds of trouble. One standard deviation more living area, about 38 m², multiplies the odds by 0.72. At the bottom of the table `rooms` (0.97), `years_client` (0.97) and the energy-label dummies (between 0.88 and 1.08, in no order from B to G) sit close to 1. A column with an odds ratio of 1 changes nothing, and this is the first hint that not every column in the file carries information about repayment. Part B returns to it with a method that selects features.

In [ ]:
# From probabilities to decisions: the 0.5 threshold, the confusion matrix and the two error rates
pred_05 = (p_logit >= 0.5).astype(int)
cm = confusion_matrix(y_test, pred_05)
print("confusion matrix (rows: actual 0/1, columns: predicted 0/1)\n", cm)
print(f"accuracy  : {accuracy_score(y_test, pred_05):.3%}   (majority baseline {1 - y_test.mean():.3%})")
print(f"recall    : {recall_score(y_test, pred_05):.3%}   share of troubled loans the model flags")
print(f"precision : {precision_score(y_test, pred_05, zero_division=0):.3%}   share of flagged loans that are troubled")

In [ ]:
# ROC curve and AUC, computed from the probabilities, never from the 0/1 predictions
fpr, tpr, thr = roc_curve(y_test, p_logit)
auc_logit = roc_auc_score(y_test, p_logit)
print(f"AUC on the test set: {auc_logit:.3f}   (coin flip: 0.500)")
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, lw=2, label=f"logit, AUC = {auc_logit:.3f}")
ax.plot([0, 1], [0, 1], "--", color="grey", label="coin flip, AUC = 0.5")
ax.set_xlabel("false positive rate (good loans flagged)"); ax.set_ylabel("true positive rate (troubled loans flagged)")
ax.set_title("ROC curve on the test set"); ax.legend()
plt.show()

In [ ]:
# The two errors have different prices. Expected cost per 1,000 applications at a given threshold:
def cost_per_1000(y_true, prob, threshold):
    pred = (np.asarray(prob) >= threshold).astype(int)
    fn = int(((pred == 0) & (np.asarray(y_true) == 1)).sum())   # approved, ran into trouble
    fp = int(((pred == 1) & (np.asarray(y_true) == 0)).sum())   # rejected, would have been fine
    return (fn * COST_FN + fp * COST_FP) / len(y_true) * 1000

def best_threshold(y_true, prob, grid=np.arange(0.02, 0.51, 0.01)):
    costs = [cost_per_1000(y_true, prob, t) for t in grid]
    return float(grid[int(np.argmin(costs))])

print(f"cost per 1,000 applications at threshold 0.5   : CHF {cost_per_1000(y_test, p_logit, 0.5):,.0f}")
print(f"cost per 1,000 applications approving everyone : CHF {cost_per_1000(y_test, p_logit, 1.01):,.0f}")

At the conventional threshold of 0.5 the model declines an application only when it thinks trouble is more likely than not, and in this run that is almost nobody: 48 of the 3,000 test applications, of which 28 did run into trouble and 20 would have been fine. The other 180 troubled loans pass. Recall is 13.5 %, precision 58.3 %, and accuracy is 93.3 % against 93.1 % for the baseline that approves everyone, which is the number the CRO must never be shown on its own. The ROC curve tells a different story. The AUC is the probability that a randomly chosen troubled loan receives a higher predicted probability than a randomly chosen good one, so 0.5 is a coin flip and 1 is a perfect ranking. An AUC of 0.816 in this run means that the model's ranking of the applications carries real information, and the 0.5 threshold throws most of it away.

The costs say the same in francs. At 0.5 the expected cost is CHF 3,653,333 per 1,000 applications, against CHF 4,160,000 for approving everyone. Almost all of it, CHF 3.6 million, is the 180 approved loans that went into trouble; the 20 rejected good loans add CHF 53,333. The reason is the CRO's price list. A troubled loan costs CHF 60,000 and a rejected good one CHF 8,000, so a troubled loan costs 7.5 times a lost customer. Declining pays as soon as the expected loss from approving, the probability of trouble times CHF 60,000, exceeds the expected loss from declining, the probability that the loan would have been fine times CHF 8,000. That happens at a probability of trouble of 8,000 / 68,000, about 12 %. The bar belongs well below 0.5, and Exercise 3 finds it empirically by sweeping the threshold.

One deliberate difference from the logistic-regression class: there we undersampled the majority class of the training set as a teaching device, so that the model saw a balanced world, and corrected the intercept afterwards to get deployment probabilities back. Here we do not. The threshold is chosen in CHF, and a cost calculation needs probabilities that mean what they say: a predicted 14 % must be a 14 % chance of trouble, which a fit on the book's own base rate gives directly, without a correction that is only approximate. With 693 troubled loans in the book there is enough to learn from, and the imbalance is handled where it belongs, at the threshold.

What the analyst tells the CRO: a logistic regression on the eighteen columns known at decision time ranks applications with an AUC of 0.816 on unseen loans, the affordability ratio, the loan-to-value ratio and a variable rate are the strongest signals (income enters only as a proxy for exposure at fixed ratios), and at the textbook bar of 0.5 it would decline almost nobody and cost the bank nearly as much as approving everyone. Because a troubled loan costs 7.5 times a lost good customer, the approval bar belongs near a predicted probability of trouble of 12 % rather than 50 %, and Exercise 3 sets it in francs.

### Exercise 3: The approval bar in francs

Sweep the threshold from 0.02 to 0.50 in steps of 0.01 with `cost_per_1000`, plot the expected cost per 1,000 applications against the threshold, and report the cost-minimising threshold together with the recall, precision and cost at that threshold. (This sweep uses the test set; Part B does it properly on cross-validated training predictions.)

*Deliverable:* the threshold, the three numbers, and one sentence for the CRO on what the bar means: at which predicted probability of trouble does the bank stop approving?

In [ ]:
# Exercise 3: your code here
# grid = np.arange(0.02, 0.51, 0.01)

### Exercise 4: The Swiss rule, checked on our own book

The affordability rule says the imputed cost may not exceed one third of gross income, and the standard maximum loan-to-value is 80 %. On the full book, compute the trouble rate for mortgages above and below the 1/3 affordability line, above and below 80 % LTV, and for the four combinations of the two.

*Deliverable:* a 2×2 table of trouble rates (affordability above/below 1/3 by LTV above/below 0.8) and one sentence on what it shows. Keep this table in mind for Part B.

In [ ]:
# Exercise 4: your code here
# over_aff = book.affordability > 1/3

# Part B: The advanced methods

The CRO's third request is the one the risk committee is most curious about and the one the CRO trusts least. The committee has read that banks use "machine learning" for credit decisions. The CRO wants to know two things: whether the more flexible methods actually do better on this book than the logistic regression of Part A2, and whether the bank could defend deploying one, to its supervisor and to a customer whose application it declined. Both halves count. A model that gains a few points of AUC and cannot be explained to anyone is not an improvement for a bank.

Part B fits five methods from the advanced supervised-learning class one after the other, LASSO, a decision tree, a random forest, gradient boosting and a support vector machine, and then puts all of them in one table next to the logistic regression. Every method is trained on the same 7,000 mortgages and judged on the same 3,000 held-out ones, with the same two numbers as in Part A2, the AUC of its ranking and the cost in CHF of its decisions; a comparison on anything else would not be fair. Each method is introduced by what it does differently from the logit, in the words of the class. LASSO keeps the logit but adds a penalty that shrinks the coefficients and sets some to exactly zero, so it selects the columns. A decision tree asks one yes/no question at a time and can therefore draw the corner that Exercise 4 found. A random forest is a panel of trees, each grown on its own sample of the data and of the columns, that votes. Boosting grows trees that correct each other's mistakes, and a support vector machine looks for the widest margin between the two classes. Part C then hands the same tools to your team.

## B1. LASSO: let the data choose the columns

Part A2 ended with a hint. `rooms`, `years_client` and the energy labels had odds ratios close to 1, and the analyst had chosen the columns by hand. Here we do the opposite and give the model more columns than it needs. To the eighteen features of Part A2 we add four: the `overpayment` ratio of Exercise 2 (recomputed in the cell so that nothing here depends on your exercise code), the logarithm of household income, and two derived columns that sound plausible in a credit meeting, income per room and price per square metre. Some of these carry information, some duplicate a column that is already in, and some are noise. The point is that we do not have to know which in advance.

A plain logistic regression with many columns spreads small coefficients over everything, noise included, and every coefficient it estimates costs precision. LASSO changes the objective. It fits the same logistic regression but adds a penalty λ·Σ|β| on the sum of the absolute coefficients, so that fitting the training data well is traded against keeping the coefficients small. Because the penalty uses absolute values, coefficients do not merely shrink as λ grows; they reach exactly zero and stay there, and a column with a zero coefficient has left the model. The model selects. Ridge regression penalises Σβ² instead, which shrinks every coefficient but never sets one to zero; elastic net mixes the two penalties. scikit-learn writes the strength of the penalty as C = 1/λ, so a *small* C means a *strong* penalty and few surviving columns, and a large C means almost no penalty and the plain logit back. The penalty treats every coefficient alike, so the columns must be on the same scale; the LASSO is fitted on standardised columns, as the logit was.

The first cell builds the wide feature set. The second traces the LASSO path: the model is refitted for thirty values of C from a strong penalty to almost none, and the plot shows each coefficient entering as the penalty relaxes, with the columns that end up large labelled. The third cell lets five-fold cross-validation on the training set choose C by AUC, and reads off which columns survived.

In [ ]:
# A wider feature set: the valuation residual from Part A1 (recomputed here so this cell does not depend on Exercise 2)
lin_log_b = LinearRegression().fit(Xp_train, np.log(yp_train))
book["overpayment"] = book.purchase_price / np.exp(lin_log_b.predict(Xp))
book["log_income"] = np.log(book.household_income)
book["income_per_room"] = book.household_income / book.rooms
book["price_per_m2"] = book.purchase_price / book.living_area_m2

wide_features = features + ["overpayment", "log_income", "income_per_room", "price_per_m2"]
Xw = pd.get_dummies(book[wide_features], drop_first=True).astype(float)
Xw_train, Xw_test = Xw.loc[X_train.index], Xw.loc[X_test.index]     # same rows as before
scaler_w = StandardScaler().fit(Xw_train)
Xw_train_s = pd.DataFrame(scaler_w.transform(Xw_train), columns=Xw.columns, index=Xw_train.index)
Xw_test_s = pd.DataFrame(scaler_w.transform(Xw_test), columns=Xw.columns, index=Xw_test.index)
print(Xw.shape[1], "columns:", list(Xw.columns))

In [ ]:
# The LASSO path: refit for a grid of C (= 1/penalty) and watch the coefficients shrink to zero one by one
import warnings   # penalty="l1" is deprecated from scikit-learn 1.8 and removed in 1.10 (use l1_ratio=1 then); silence only that
warnings.filterwarnings("ignore", message="'penalty' was deprecated")
warnings.filterwarnings("ignore", message="Inconsistent values: penalty")
Cs = np.logspace(-3, 1, 30)
path = np.array([LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=2000)
                 .fit(Xw_train_s, y_train).coef_[0] for C in Cs])

fig, ax = plt.subplots(figsize=(9, 5.5))
for j, name in enumerate(Xw.columns):
    ax.plot(Cs, path[:, j], lw=1.5, label=name if np.abs(path[-1, j]) > 0.15 else None)
ax.set_xscale("log"); ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("C = 1 / penalty strength  (left: strong penalty, right: almost none)")
ax.set_ylabel("coefficient (log-odds per SD)")
ax.set_title("LASSO path: which columns survive as the penalty relaxes"); ax.legend(fontsize=8, ncol=2)
plt.show()
n_nonzero = (np.abs(path) > 1e-6).sum(axis=1)
print(pd.DataFrame({"C": Cs.round(4), "non-zero coefficients": n_nonzero}).iloc[::3].to_string(index=False))

In [ ]:
# Let cross-validation choose C, then read the survivors
warnings.filterwarnings("ignore", message="The default value for l1_ratios")            # two more scikit-learn 1.8 -> 1.10
warnings.filterwarnings("ignore", message="The fitted attributes of LogisticRegressionCV")  # transition notices, nothing else
lasso = LogisticRegressionCV(Cs=Cs, penalty="l1", solver="liblinear", cv=5, scoring="roc_auc",
                             max_iter=2000, random_state=0).fit(Xw_train_s, y_train)
p_lasso = lasso.predict_proba(Xw_test_s)[:, 1]
survivors = pd.Series(lasso.coef_[0], index=Xw.columns)
print(f"chosen C: {lasso.C_[0]:.4f}   |   {int((survivors.abs() > 1e-6).sum())} of {len(survivors)} coefficients non-zero")
print("dropped:", list(survivors.index[survivors.abs() <= 1e-6]))
print(f"test AUC  logit (Part A2 features): {auc_logit:.3f}   LASSO (wide features): {roc_auc_score(y_test, p_lasso):.3f}")
cv_auc = lasso.scores_[1].mean(axis=0)                     # mean cross-validated AUC for each C in Cs, the number CV chose on
i_small, i_best = int(np.abs(Cs - 0.045).argmin()), int(np.argmax(cv_auc))
print(f"mean CV AUC  at C = {Cs[i_small]:.3f}: {cv_auc[i_small]:.4f}   at the chosen C = {Cs[i_best]:.3f}: {cv_auc[i_best]:.4f}")
results = {"logit": (logit, X_train_s, X_test_s), "LASSO logit": (lasso, Xw_train_s, Xw_test_s)}
survivors[survivors.abs() > 1e-6].sort_values(key=abs, ascending=False).round(3).to_frame("coef")

In this run the path table is a story read from left to right. Under a strong penalty, C = 0.001, every coefficient is zero and the model predicts the same number for everyone. The first column to enter, at C = 0.003, is `affordability`; `ltv`, `rate_type_saron` and the two employment dummies follow, and by C = 0.045 thirteen columns are in. Only at the far right, at C = 5.3, has every one of the 33 columns a non-zero coefficient, which is the plain logit. Cross-validation chose C = 0.161. There 26 of the 33 columns survive and seven are dropped: `rooms`, `living_area_m2`, `canton_BE`, `property_type_house`, and three of the four columns we added, `log_income`, `income_per_room` and `price_per_m2`. The three dropped additions duplicate information that is already in the model, log income above all, which is the income column on another scale. The fourth addition, `overpayment`, survived with the fifth largest coefficient, 0.300 per standard deviation, and it explains two changes at the top of the table. In Part A2 living area was protective and income raised the odds at fixed ratios; here living area is gone and the income coefficient has fallen to 0.033. A large loan on a small property relative to income was the logit's way of saying that the buyer paid more than the property is worth, and now the column that says so directly is in the model. A team that removes `overpayment` from `wide_features` and reruns the two cells will see the income coefficient come back.

The selection is not sharp, and that is the second lesson of the table. `years_client` survived with a coefficient of -0.016, and five of the six energy-label dummies with coefficients between -0.017 and 0.021 (only label G, at -0.098, is larger), because at C = 0.161 the penalty is mild. The path table shows thirteen columns at C = 0.045, and the cross-validated AUC there, 0.8212, is within a hair of the 0.8213 at the chosen C: cross-validation found almost nothing to choose between thirteen columns and twenty-six. LASSO tells the analyst which columns can go without changing the ranking; it does not make the model more flexible. The test AUC says the same: 0.823 against the logit's 0.816, less than one point for four new columns and a penalty, and the corner from Exercise 4 is as far beyond a penalised logit as it was beyond the plain one.

What the analyst tells the CRO: given 33 columns, the LASSO kept 26 and discarded the ones that duplicate others, and its ranking of unseen applications is as good as the logistic regression's (an AUC of 0.823 against 0.816 on the same loans), with the same signals near the top, loan-to-value, affordability and a variable rate, plus one new one, a purchase price above our own valuation of the property. LASSO makes the model smaller, not more flexible; if the flexible methods are to earn their place, it will be the next two.

## B2. A decision tree: rules a credit officer can read

Exercise 4 found something the logit cannot say. Mortgages that breach both the affordability rule and the 80 % loan-to-value line have a trouble rate of about 41 %, against 3 to 6 % in the other three cells of the table, so the two ratios do not add, they multiply. A logistic regression with one log-odds term per column can raise the risk along each axis but cannot draw a corner. This is the question the class asked after the linear models, "what about non-linear relationships?"

A decision tree answers it by asking one yes/no question at a time. It looks for the single column and the single threshold that best separate the troubled mortgages from the fine ones, splits the training set in two, and then asks the next question separately in each half. In the class this was the Titanic tree, where two questions, sex and age, were enough to isolate the passengers who did not survive. Two thresholds on two different columns are exactly a corner. Each leaf of the tree holds a group of mortgages and its share of trouble, which is the tree's predicted probability for every application that lands there, and the path from the root to a leaf is a rule that a credit officer can read out and apply by hand.

Two settings control how many questions the tree may ask. `max_depth` limits the number of questions on any path from the root to a leaf, and `min_samples_leaf` refuses any split that would leave a leaf with fewer than that many mortgages. Both exist to stop the tree from memorising the training set, which a tree left to itself will do until every leaf holds a single loan; the third cell below shows what that looks like. A tree compares each value with a threshold and does not care about scale, so it is fitted on the raw dummy columns of Part A2 rather than the standardised ones.

The first cell grows a tree of depth three, draws it, and prints the same tree as text. The second draws the empirical trouble rate over the two ratios as a heat map, which is where the tree looks. The third grows trees of depth 1 to 12 and puts the training AUC next to the test AUC for each.

In [ ]:
# A shallow tree: three questions deep, at least 50 mortgages per leaf
tree3 = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=0).fit(X_train, y_train)
p_tree = tree3.predict_proba(X_test)[:, 1]
print(f"test AUC  tree (depth 3): {roc_auc_score(y_test, p_tree):.3f}   logit: {auc_logit:.3f}")

fig, ax = plt.subplots(figsize=(20, 9))
plot_tree(tree3, feature_names=list(X.columns), class_names=["fine", "trouble"], filled=True,
          impurity=False, proportion=True, rounded=True, fontsize=10, ax=ax)
plt.show()
# The same tree as text: each line is one question, each leaf shows the class it predicts
print(export_text(tree3, feature_names=list(X.columns), show_weights=True, decimals=3))
results["tree (depth 3)"] = (tree3, X_train, X_test)

In [ ]:
# Where the tree looks: the empirical trouble rate over LTV and affordability, with the two rule lines drawn in
grid_ltv = pd.cut(book.ltv, bins=[0.3, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.91])
grid_aff = pd.cut(book.affordability, bins=[0, 0.2, 0.25, 0.3, 1/3, 0.4, 0.5, 2.0])
heat = book.pivot_table(index=grid_aff, columns=grid_ltv, values="trouble_36m", aggfunc="mean", observed=False)
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.heatmap(heat * 100, annot=True, fmt=".0f", cmap="Reds", cbar_kws={"label": "trouble rate (%)"}, ax=ax)
ax.axvline(5, color="black", lw=1.5); ax.axhline(4, color="black", lw=1.5)   # the 80 % LTV line and the one-third line
ax.invert_yaxis()
ax.set_xlabel("loan-to-value"); ax.set_ylabel("affordability ratio")
ax.set_title("Trouble rate (%) by LTV and affordability: the corner the logit cannot draw")
plt.show()

In [ ]:
# Why we limit the depth: deeper trees memorise the training set
depths = range(1, 13)
auc_tr, auc_te = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, min_samples_leaf=5, random_state=0).fit(X_train, y_train)
    auc_tr.append(roc_auc_score(y_train, t.predict_proba(X_train)[:, 1]))
    auc_te.append(roc_auc_score(y_test, t.predict_proba(X_test)[:, 1]))
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(depths, auc_tr, marker="o", label="training AUC"); ax.plot(depths, auc_te, marker="o", label="test AUC")
ax.set_xlabel("max_depth"); ax.set_ylabel("AUC"); ax.set_title("Overfitting: the gap opens as the tree deepens"); ax.legend()
plt.show()
print(pd.DataFrame({"depth": depths, "train AUC": np.round(auc_tr, 3), "test AUC": np.round(auc_te, 3)}).to_string(index=False))

In this run the tree's first question is the Swiss loan-to-value rule: is `ltv` at most 0.801? The threshold is the tree's own choice, and it landed on the bank's 80 % line. Of the 7,000 training mortgages, 5,930 answer yes and 3.8 % of them ran into trouble; 1,070 answer no and 24.5 % did. The second question on the high-LTV side is the affordability rule, again with the threshold found by the tree, 0.339, which is the one-third line: 488 mortgages below it with 4.7 % trouble, 582 above it with 41.1 %. That is the corner of Exercise 4, recovered from the data in two questions. The third question inside the corner is `fixed_years <= 2.5`, which in this book means "is it a SARON mortgage", because every fixed-rate mortgage has a term of 5 or 10 years and every SARON mortgage a term of 0. The leaf with the highest trouble share is that one: LTV above 80 %, affordability above one third, variable rate, 203 training mortgages of which 119, or 58.6 %, ran into trouble, and it is the only leaf the tree labels `trouble`. Its fixed-rate neighbour holds 379 mortgages with 31.7 %. On the low-LTV side the tree finds a second, smaller pocket, SARON mortgages whose actual interest burden exceeds 0.095 of income, 69 mortgages with 46.4 % trouble, and among fixed-rate borrowers it separates those over 62 (112 mortgages, 13.4 %) from everyone else, the 3,768 mortgages with 1.6 % trouble where most of the book lives. Three questions, eight leaves, and a test AUC of 0.824 against the logit's 0.816.

The heat map is the same corner without a model. Below 80 % LTV the cells read between 1 and 9 % up to an affordability of 0.5 and 9 to 14 % above it, apart from the left-most column, where the book holds very few loans and a handful of cases move the percentage. Above 80 % LTV but within the affordability rule the cells read 2 to 9 %. Above 80 % LTV and above one-third affordability, the six cells read 33 to 50 %. A logistic regression can raise the risk along each axis; it cannot make those six cells different in kind from their neighbours. Two thresholds do.

The depth sweep shows the price of asking more questions. Training and test AUC rise together up to depth 4, where the test AUC peaks at 0.854 with a training AUC of 0.853. From depth 5 the two part: the training AUC keeps climbing, to 0.971 at depth 12, while the test AUC falls to 0.849, then 0.844, is back at the logit's level by depth 7 (0.816) and ends at 0.647. A tree of depth 12 with five loans per leaf has memorised the training set and has lost most of its edge on new loans. The depth-3 tree gives up three points of AUC against the best depth in exchange for eight leaves that fit on one page.

What the analyst tells the CRO: a decision tree of three questions, fitted on the same columns as the logistic regression, rediscovers the bank's own two rules, 80 % loan-to-value and one-third affordability, as its first two questions, adds the rate type as the third, and ranks unseen applications slightly better than the logistic regression (an AUC of 0.824 against 0.816). Its eight leaves are rules a credit officer can apply by hand, and the one that matters says that a variable-rate mortgage that breaches both rules went wrong in 59 of 100 cases in our book. It is the first model in this notebook the officers could run without a computer. Deeper trees memorise the book and do worse on new loans, so if the bank wants a tree, it wants a shallow one.

## B3. A random forest: a panel instead of one interviewer

The class introduced the forest with a hiring analogy. A single tree is one interviewer: it asks its questions in a fixed order, and a slightly different set of candidates would have led it to open with a different question and to reach a different verdict. That instability is where the overfitting in the depth plot comes from. A random forest replaces the interviewer with a panel. Each of several hundred trees is grown on a bootstrap sample of the training set, drawn with replacement, so that every tree sees some mortgages twice and others not at all, and at each question the tree may only choose among a random subset of the qualifications, so that not every tree opens with loan-to-value, as the single tree in B2 did. Every tree is grown deep. The forest's prediction for an application is the verdict of the panel. In the slide's picture each tree votes trouble or fine and the majority wins; what scikit-learn computes is one step finer: each tree reports the trouble share of the leaf the application lands in, and the forest averages those shares, which is the same as the vote share when every leaf is pure and a smoother number when, as here, leaves hold at least five loans. Each tree memorises its own sample, but the trees memorise different things, and the average washes the memorising away. That is why a forest generalises where one deep tree does not.

The price is readability. Nobody can read three hundred trees, and there is no table of coefficients. What we can ask the forest is which columns it *used*. Permutation importance shuffles one column of the test set, so that this column becomes noise while everything else stays as it was, and records how far the AUC drops; the shuffle is repeated a few times and the drops averaged. A column the forest relies on costs points of AUC when it is shuffled; a column the forest ignores costs nothing.

The first cell grows forests of 1, 5, 25, 100 and 300 trees, records the test AUC of each, and keeps the panel of 300. The second asks that panel which columns matter.

In [ ]:
# How many interviewers does the panel need? Test AUC against the number of trees
n_grid = [1, 5, 25, 100, 300]
auc_n = []
for n in n_grid:
    f = RandomForestClassifier(n_estimators=n, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(X_train, y_train)
    auc_n.append(roc_auc_score(y_test, f.predict_proba(X_test)[:, 1]))
print(pd.DataFrame({"trees": n_grid, "test AUC": np.round(auc_n, 3)}).to_string(index=False))

rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(X_train, y_train)
p_rf = rf.predict_proba(X_test)[:, 1]
print(f"\ntraining AUC of the forest: {roc_auc_score(y_train, rf.predict_proba(X_train)[:, 1]):.3f}   test AUC: {roc_auc_score(y_test, p_rf):.3f}")
results["random forest"] = (rf, X_train, X_test)

In [ ]:
# Which columns does the forest rely on? Permutation importance on the test set (drop in AUC when a column is shuffled)
imp = permutation_importance(rf, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=0, n_jobs=-1)
imp_s = pd.Series(imp.importances_mean, index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 5))
imp_s.head(10)[::-1].plot.barh(ax=ax)
ax.set_xlabel("drop in test AUC when the column is shuffled"); ax.set_title("Random forest: the ten columns that matter most")
plt.show()
print(imp_s.head(10).round(4).to_string())

In this run one interviewer is a poor judge and a panel is a good one. A single deep tree scores a test AUC of 0.706, five trees 0.807, twenty-five 0.850, and from there the panel gains little: 0.857 with 100 trees and 0.858 with 300. The jump from one tree to twenty-five is the vote at work. Each tree is grown until its leaves hold at least five loans and memorises its own bootstrap sample, and the forest's training AUC of 0.989 shows that the memorising is still there. What the vote removes is its effect on new loans, where the same forest scores 0.858, above the logit (0.816), the LASSO (0.823), the depth-3 tree (0.824) and the best single tree of the depth sweep (0.854 at depth 4). A model that fits the training set almost perfectly and still generalises is something the first two classes had no example of, and it is what the committee has read about.

The permutation importances say that the forest relies on the columns the tree and the logit found. Shuffling `ltv` costs 0.083 of test AUC, shuffling `affordability` 0.046 and shuffling `age` 0.031; `employment_self_employed` follows with 0.027, then `fixed_years` (0.011) and `rate_type_saron` (0.009). Those last two carry the same information, since a SARON mortgage is one with `fixed_years` of 0, so the forest spreads the rate-type signal over both columns and each looks smaller than the pair; shuffling one leaves the other to answer. Below the top six the drops are 0.007 for `actual_burden` and at most 0.002 for the rest, `rooms` and `household_income` among them. Age is the one column that ranks higher here than in the linear models, where it carried a small coefficient (an odds ratio of 0.92 per standard deviation in Part A2). The tree showed why: the risk in age is a step at about 62 rather than a slope, which a linear term cannot express and a forest of trees can.

What the analyst tells the CRO: a random forest of 300 trees ranks unseen applications with an AUC of 0.858, four points above the logistic regression on the same columns and the same test loans, and it does so by relying on the same signals the readable models found, loan-to-value, affordability, age, self-employment and the rate type; there is no hidden column in it. What it lacks is a rule. Nobody can read 300 trees, so a declined customer would be shown the importance chart rather than the question that declined them. Whether four points of AUC are worth that is a question in francs, and the comparison table at the end of Part B puts a number on it.